In [1]:
ENV["CLIMACOMMS_DEVICE"] = "CUDA"
ENV["CLIMA_NAME_CUDA_KERNELS_FROM_STACK_TRACE"] = "true"

using ClimaCore: Geometry, Operators, MatrixFields
import ClimaCore
import LazyBroadcast: lazy

# Alternatively, we could use Vec₁₂₃, Vec³, etc., if that is more readable.
const C1 = Geometry.Covariant1Vector
const C2 = Geometry.Covariant2Vector
const C12 = Geometry.Covariant12Vector
const C3 = Geometry.Covariant3Vector
const C123 = Geometry.Covariant123Vector
const CT1 = Geometry.Contravariant1Vector
const CT2 = Geometry.Contravariant2Vector
const CT12 = Geometry.Contravariant12Vector
const CT3 = Geometry.Contravariant3Vector
const CT123 = Geometry.Contravariant123Vector
const UVW = Geometry.UVWVector

const divₕ = Operators.Divergence()
const wdivₕ = Operators.WeakDivergence()
const split_divₕ = Operators.SplitDivergence()
const gradₕ = Operators.Gradient()
const wgradₕ = Operators.WeakGradient()
const curlₕ = Operators.Curl()
const wcurlₕ = Operators.WeakCurl()

const ᶜinterp = Operators.InterpolateF2C()
const ᶜdivᵥ = Operators.DivergenceF2C()
const ᶜgradᵥ = Operators.GradientF2C()

# Tracers do not have advective fluxes through the top and bottom cell faces.
const ᶜadvdivᵥ = Operators.DivergenceF2C(
    bottom=Operators.SetValue(CT3(0)),
    top=Operators.SetValue(CT3(0)),
)

# Subsidence has extrapolated tendency at the top, and has no flux at the bottom.
# TODO: This is not accurate and causes some issues at the domain top.
const ᶜsubdivᵥ = Operators.DivergenceF2C(
    bottom=Operators.SetValue(CT3(0)),
    top=Operators.Extrapolate(),
)

# Precipitation has no flux at the top, but it has free outflow at the bottom.
const ᶜprecipdivᵥ = Operators.DivergenceF2C(top=Operators.SetValue(CT3(0)))

const ᶠright_bias = Operators.RightBiasedC2F() # for free outflow in ᶜprecipdivᵥ
const ᶜleft_bias = Operators.LeftBiasedF2C()
const ᶜright_bias = Operators.RightBiasedF2C()

# TODO: Implement proper extrapolation instead of simply reusing the first
# interior value at the surface.
const ᶠinterp = Operators.InterpolateC2F(
    bottom=Operators.Extrapolate(),
    top=Operators.Extrapolate(),
)
const ᶠwinterp = Operators.WeightedInterpolateC2F(
    bottom=Operators.Extrapolate(),
    top=Operators.Extrapolate(),
)

# TODO: Replace these boundary conditions with NaN's, since they are
# meaningless and we only need to specify them in order to be able to
# materialize broadcasts. Any effect these boundary conditions have on the
# boundary values of Y.f.u₃ is overwritten when we call set_velocity_at_surface!.
# Ideally, we would enforce the boundary conditions on Y.f.u₃ by filtering it
# immediately after adding the tendency to it. However, this is not currently
# possible because our implicit solver is unable to handle filtering, which is
# why these boundary conditions are 0's rather than NaN's.
const ᶠgradᵥ = Operators.GradientC2F(
    bottom=Operators.SetGradient(C3(0)),
    top=Operators.SetGradient(C3(0)),
)
const ᶠcurlᵥ = Operators.CurlC2F(
    bottom=Operators.SetCurl(CT12(0, 0)),
    top=Operators.SetCurl(CT12(0, 0)),
)
const upwind_biased_grad = Operators.UpwindBiasedGradient()
const ᶠupwind1 = Operators.UpwindBiasedProductC2F()
const ᶠupwind3 = Operators.Upwind3rdOrderBiasedProductC2F(
    bottom=Operators.ThirdOrderOneSided(),
    top=Operators.ThirdOrderOneSided(),
)
@static if pkgversion(ClimaCore) ≥ v"0.14.22"
    const ᶠlin_vanleer = Operators.LinVanLeerC2F(
        bottom=Operators.FirstOrderOneSided(),
        top=Operators.FirstOrderOneSided(),
        constraint=Operators.MonotoneLocalExtrema(), # (Mono5)
    )
end

const ᶜinterp_matrix = MatrixFields.operator_matrix(ᶜinterp)
const ᶜleft_bias_matrix = MatrixFields.operator_matrix(ᶜleft_bias)
const ᶜright_bias_matrix = MatrixFields.operator_matrix(ᶜright_bias)
const ᶜdivᵥ_matrix = MatrixFields.operator_matrix(ᶜdivᵥ)
const ᶜadvdivᵥ_matrix = MatrixFields.operator_matrix(ᶜadvdivᵥ)
const ᶜprecipdivᵥ_matrix = MatrixFields.operator_matrix(ᶜprecipdivᵥ)
const ᶠright_bias_matrix = MatrixFields.operator_matrix(ᶠright_bias)
const ᶠinterp_matrix = MatrixFields.operator_matrix(ᶠinterp)
const ᶠwinterp_matrix = MatrixFields.operator_matrix(ᶠwinterp)
const ᶠgradᵥ_matrix = MatrixFields.operator_matrix(ᶠgradᵥ)
const ᶠupwind1_matrix = MatrixFields.operator_matrix(ᶠupwind1)
const ᶠupwind3_matrix = MatrixFields.operator_matrix(ᶠupwind3)

# Helper functions to extract components of vectors
u_component(u::Geometry.LocalVector) = u.u
v_component(u::Geometry.LocalVector) = u.v
w_component(u::Geometry.LocalVector) = u.w

include(
    joinpath(pkgdir(ClimaCore), "test", "TestUtilities", "TestUtilities.jl"),
);
import .TestUtilities as TU;
using Test
using ClimaComms
ClimaComms.@import_required_backends
using StaticArrays, IntervalSets
import BenchmarkTools
import StatsBase
import DataStructures
using ClimaCore.Geometry: ⊗
import ClimaCore.DataLayouts

FT = Float32
horizontal_layout_type = ClimaCore.DataLayouts.IJFH
z_elems = 63
helem = 30
Nq = 4
cspace = TU.CenterExtrudedFiniteDifferenceSpace(FT; zelem=z_elems, helem, Nq, horizontal_layout_type)
fspace = ClimaCore.Spaces.FaceExtrudedFiniteDifferenceSpace(cspace)
import ClimaCore: Domains, Meshes, Spaces, Fields, Operators, Topologies
import ClimaCore.Domains: Geometry

using BenchmarkTools
field_vars(::Type{FT}) where {FT} = (;
    x=FT(1),
    uₕ=Geometry.Covariant12Vector(FT(0), FT(0)),
    uₕ2=Geometry.Covariant12Vector(FT(0), FT(0)),
    curluₕ=Geometry.Contravariant12Vector(FT(0), FT(0)),
    w=Geometry.Covariant3Vector(FT(0)),
    contra3=Geometry.Contravariant3Vector(FT(0)),
    y=FT(1),
    D=FT(0),
    U=FT(0),
    ∇x=Geometry.Covariant3Vector(FT(0)),
    ᶠu³=Geometry.Contravariant3Vector(FT(0)),
    ᶠuₕ³=Geometry.Contravariant3Vector(FT(1)),
    ᶠw=Geometry.Covariant3Vector(FT(0)),
)
cfield = fill(field_vars(FT), cspace)
cfield2 = fill(field_vars(FT), cspace)
ffield = fill(field_vars(FT), fspace)
# @. ᶠlin_vanleer(ffield.ᶠu³, cfield.x, 0.1f0)
ᶜJ = Fields.local_geometry_field(axes(cfield)).J
ᶠJ = Fields.local_geometry_field(axes(ffield)).J

┌ Warning: CUDA runtime library `libcublasLt.so.13` was loaded from a system path, `/usr/local/cuda/lib64/libcublasLt.so.13`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218
┌ Warning: CUDA runtime library `libnvJitLink.so.13` was loaded from a system path, `/usr/local/cuda/lib64/libnvJitLink.so.13`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218


┌ Warning: CUDA runtime library `libcusparse.so.12` was loaded from a system path, `/usr/local/cuda/lib64/libcusparse.so.12`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218


Float32-valued Field:
  Float32[0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472  …  0.132472, 0.132471, 0.132472, 0.132471, 0.132472, 0.132471, 0.132472, 0.132471, 0.132472, 0.132472]

In [2]:
bi_fs = (fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace))

bi_cs = (fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace))

di_fs = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace))

di_cs = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace))

di_cs2 = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace))

di_cs3 = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace))

tri = fill(ClimaCore.MatrixFields.TridiagonalMatrixRow(0.0f0, 0.0f0, 0.0f0), fspace)
tri2 = fill(ClimaCore.MatrixFields.TridiagonalMatrixRow(0.0f0, 0.0f0, 0.0f0), cspace)

ClimaCore.MatrixFields.TridiagonalMatrixRow{Float32}-valued Field whose first column corresponds to the Square matrix
 0.0  0.0   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   …   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
 0.0  0.0  0.0   ⋅    ⋅    ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅   0.0  0.0  0.0   ⋅    ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅   0.0  0.0  0.0   ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅   0.0  0.0  0.0   ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅   0.0  0.0  0.0   ⋅   …   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅   0.0  0.0  0.0      ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   0.0  0.0      ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   0.0      ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   …   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅ 

In [3]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1])) - tri
end

Profiler ran for 4.12 s, capturing 71 events.

Host-side activity: calling CUDA APIs took 798.94 µs (0.02% of the trace)
┌────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│ ID │     Start │      Time │ Name                │                   Details │
├────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│  6 │ 598.75 ms │ 594.38 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 55 │    4.12 s │    9.3 µs │ cuCtxSynchronize    │                         - │
│ 59 │    4.12 s │  76.06 µs │ cuModuleLoadDataEx  │                         - │
│ 64 │    4.12 s │  43.63 µs │ cuModuleGetFunction │                         - │
│ 68 │    4.12 s │  28.61 µs │ cuLaunchKernel      │                         - │
└────┴───────────┴───────────┴─────────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 323.53 µs (0.01% of the trace)
┌────┬────────┬───────────┬─────────┬──────────┬──────┬────────────────────────

In [4]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])) - tri
end

Profiler ran for 4.72 s, capturing 79 events.

Host-side activity: calling CUDA APIs took 747.68 µs (0.02% of the trace)
┌────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│ ID │     Start │      Time │ Name                │                   Details │
├────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│  6 │ 451.81 ms │  379.8 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 63 │    4.72 s │  28.13 µs │ cuCtxSynchronize    │                         - │
│ 67 │    4.72 s │ 123.02 µs │ cuModuleLoadDataEx  │                         - │
│ 72 │    4.72 s │ 130.89 µs │ cuModuleGetFunction │                         - │
│ 76 │    4.72 s │  41.96 µs │ cuLaunchKernel      │                         - │
└────┴───────────┴───────────┴─────────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 687.84 µs (0.01% of the trace)
┌────┬────────┬───────────┬─────────┬──────────┬──────┬────────────────────────

In [5]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3])) - tri
end

Profiler ran for 7.13 s, capturing 99 events.

Host-side activity: calling CUDA APIs took 1.72 ms (0.02% of the trace)
┌────┬──────────┬───────────┬─────────────────────┬───────────────────────────┐
│ ID │    Start │      Time │ Name                │                   Details │
├────┼──────────┼───────────┼─────────────────────┼───────────────────────────┤
│  6 │ 487.2 ms │ 568.87 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 68 │   7.01 s │ 904.32 µs │ cuMemFree           │ 63.281 MiB, device memory │
│ 72 │   7.01 s │  25.75 µs │ cuMemFree           │    8 bytes, device memory │
│ 83 │   7.12 s │   9.54 µs │ cuCtxSynchronize    │                         - │
│ 87 │   7.12 s │  91.79 µs │ cuModuleLoadDataEx  │                         - │
│ 92 │   7.12 s │  65.33 µs │ cuModuleGetFunction │                         - │
│ 96 │   7.12 s │  25.75 µs │ cuLaunchKernel      │                         - │
└────┴──────────┴───────────┴─────────────────────┴───────────────────────────┘



In [6]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])) - tri
end

Profiler ran for 10.92 s, capturing 111 events.

Host-side activity: calling CUDA APIs took 1.26 ms (0.01% of the trace)
┌─────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name                │                   Details │
├─────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│   6 │ 536.71 ms │ 471.35 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│  80 │   10.77 s │ 494.48 µs │ cuMemFree           │ 63.281 MiB, device memory │
│  84 │   10.77 s │  21.22 µs │ cuMemFree           │    8 bytes, device memory │
│  95 │   10.92 s │   9.78 µs │ cuCtxSynchronize    │                         - │
│  99 │   10.92 s │ 119.45 µs │ cuModuleLoadDataEx  │                         - │
│ 104 │   10.92 s │  80.11 µs │ cuModuleGetFunction │                         - │
│ 108 │   10.92 s │  31.23 µs │ cuLaunchKernel      │                         - │
└─────┴───────────┴───────────┴─────────────────────┴──────

In [7]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5])) - tri
end

Profiler ran for 13.78 s, capturing 123 events.

Host-side activity: calling CUDA APIs took 1.57 ms (0.01% of the trace)
┌─────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name                │                   Details │
├─────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│   6 │ 559.69 ms │ 402.21 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│  92 │    13.6 s │ 718.59 µs │ cuMemFree           │ 63.281 MiB, device memory │
│  96 │    13.6 s │  81.06 µs │ cuMemFree           │    8 bytes, device memory │
│ 107 │   13.78 s │  18.84 µs │ cuCtxSynchronize    │                         - │
│ 111 │   13.78 s │ 143.29 µs │ cuModuleLoadDataEx  │                         - │
│ 116 │   13.78 s │ 124.69 µs │ cuModuleGetFunction │                         - │
│ 120 │   13.78 s │  38.39 µs │ cuLaunchKernel      │                         - │
└─────┴───────────┴───────────┴─────────────────────┴──────

In [8]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])) - tri
end

Profiler ran for 17.8 s, capturing 135 events.

Host-side activity: calling CUDA APIs took 1.41 ms (0.01% of the trace)
┌─────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name                │                   Details │
├─────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│   6 │ 575.08 ms │ 542.16 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 104 │   17.58 s │ 508.79 µs │ cuMemFree           │ 63.281 MiB, device memory │
│ 108 │   17.58 s │  26.23 µs │ cuMemFree           │    8 bytes, device memory │
│ 119 │    17.8 s │   9.78 µs │ cuCtxSynchronize    │                         - │
│ 123 │    17.8 s │ 160.22 µs │ cuModuleLoadDataEx  │                         - │
│ 128 │    17.8 s │  94.89 µs │ cuModuleGetFunction │                         - │
│ 132 │    17.8 s │  25.03 µs │ cuLaunchKernel      │                         - │
└─────┴───────────┴───────────┴─────────────────────┴───────

In [9]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])
        + (bi_fs[7] * di_cs[7] * bi_cs[7])) - tri
end

Profiler ran for 21.31 s, capturing 147 events.

Host-side activity: calling CUDA APIs took 1.61 ms (0.01% of the trace)
┌─────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name                │                   Details │
├─────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│   6 │ 660.13 ms │ 499.01 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 116 │   21.05 s │ 601.53 µs │ cuMemFree           │ 63.281 MiB, device memory │
│ 120 │   21.05 s │  79.87 µs │ cuMemFree           │    8 bytes, device memory │
│ 131 │    21.3 s │  20.03 µs │ cuCtxSynchronize    │                         - │
│ 135 │    21.3 s │ 156.16 µs │ cuModuleLoadDataEx  │                         - │
│ 140 │    21.3 s │ 147.34 µs │ cuModuleGetFunction │                         - │
│ 144 │    21.3 s │  57.94 µs │ cuLaunchKernel      │                         - │
└─────┴───────────┴───────────┴─────────────────────┴──────

In [10]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])
        + (bi_fs[7] * di_cs[7] * bi_cs[7]) + (bi_fs[8] * di_cs[8] * bi_cs[8])) - tri
end

Profiler ran for 26.17 s, capturing 159 events.

Host-side activity: calling CUDA APIs took 1.53 ms (0.01% of the trace)
┌─────┬──────────┬───────────┬─────────────────────┬───────────────────────────┐
│  ID │    Start │      Time │ Name                │                   Details │
├─────┼──────────┼───────────┼─────────────────────┼───────────────────────────┤
│   6 │ 648.0 ms │ 620.13 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 128 │  25.88 s │ 618.46 µs │ cuMemFree           │ 63.281 MiB, device memory │
│ 132 │  25.88 s │  25.75 µs │ cuMemFree           │    8 bytes, device memory │
│ 143 │  26.16 s │  10.01 µs │ cuCtxSynchronize    │                         - │
│ 147 │  26.16 s │ 101.57 µs │ cuModuleLoadDataEx  │                         - │
│ 152 │  26.16 s │  93.46 µs │ cuModuleGetFunction │                         - │
│ 156 │  26.16 s │  26.46 µs │ cuLaunchKernel      │                         - │
└─────┴──────────┴───────────┴─────────────────────┴─────────────────

In [11]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])
        + (bi_fs[7] * di_cs[7] * bi_cs[7]) + (bi_fs[8] * di_cs[8] * bi_cs[8])
        + (bi_fs[9] * di_cs[9] * bi_cs[9])) - tri
end

Profiler ran for 29.85 s, capturing 171 events.

Host-side activity: calling CUDA APIs took 1.53 ms (0.01% of the trace)
┌─────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name                │                   Details │
├─────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│   6 │ 719.61 ms │ 600.34 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│ 140 │   29.54 s │ 601.77 µs │ cuMemFree           │ 63.281 MiB, device memory │
│ 144 │   29.54 s │   21.7 µs │ cuMemFree           │    8 bytes, device memory │
│ 155 │   29.85 s │  10.49 µs │ cuCtxSynchronize    │                         - │
│ 159 │   29.85 s │ 108.24 µs │ cuModuleLoadDataEx  │                         - │
│ 164 │   29.85 s │  103.0 µs │ cuModuleGetFunction │                         - │
│ 168 │   29.85 s │  35.05 µs │ cuLaunchKernel      │                         - │
└─────┴───────────┴───────────┴─────────────────────┴──────